##### Init



In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS dbr_dev.yanquiel")
spark.sql(f"CREATE VOLUME IF NOT EXISTS dbr_dev.yanquiel.raw_data")

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType
import re
import requests
import pandas as pd

catalog = "dbr_dev"
schema = "yanquiel"
volume_path = f"/Volumes/{catalog}/{schema}/raw_data/world_population.csv"

##### Bronze layer


In [0]:
population_schema = (
    StructType()
    .add("Rank", IntegerType())
    .add("CCA3", StringType())
    .add("Country/Territory", StringType())
    .add("Capital", StringType())
    .add("Continent", StringType())
    .add("2022 Population", IntegerType())
    .add("2020 Population", IntegerType())
    .add("2015 Population", IntegerType())
    .add("2010 Population", IntegerType())
    .add("2000 Population", IntegerType())
    .add("1990 Population", IntegerType())
    .add("1980 Population", IntegerType())
    .add("1970 Population", IntegerType())
    .add("Area (km²)", IntegerType())
    .add("Density (per km²)", DoubleType())
    .add("Growth Rate", DoubleType())
    .add("World Population Percentage", DoubleType())
)

def clean_col_name(name):
    return re.sub(r"[^0-9a-zA-Z_]", "_", name).strip("_")

df_bronze = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(population_schema)
    .load(volume_path)
)
for old_name in df_bronze.columns:
    new_name = clean_col_name(old_name)
    if new_name != old_name:
        df_bronze = df_bronze.withColumnRenamed(old_name, new_name)

df_bronze.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.brz_world_population")

response = requests.get("https://countries.dev/countries", timeout=15)
response.raise_for_status()
data = response.json()

df_bronze_api = spark.createDataFrame(pd.DataFrame(data))
df_bronze_api.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.brz_countries_api")

print(f"Bronze OK - population: {df_bronze_pop.count()} rows, API: {df_bronze_api.count()} rows")